# Практикум: множества и словари

Python, 9 класс (углублённый). Раздел 2.1, пара 2.

Двенадцать задач тремя уровнями:

- **Уровень 1 (задачи 1–6)** — обязательный. По одной идее на задачу.
- **Уровень 2 (задачи 7–10)** — на сочетание идей.
- **Уровень 3 (задачи 11–12)** — со звёздочкой.

Если успели только первый уровень — это нормальный результат за пару.

Пишите код вместо `# ваш код здесь` и запускайте ячейку: если все `assert` проходят, внизу
печатается «ок». В нескольких задачах проверяется не только ответ, но и **способ**: решение
не должно искать перебором по списку там, где можно спросить множество. Для этого в тестах
используется `Probe` — число в обёртке, которое считает, сколько раз его сравнили.
Он определён в первой ячейке, её надо запустить.

In [15]:
# готовые инструменты для проверок — писать их не нужно, просто запустите ячейку
import random


class Probe:
    # число в обёртке, которое считает, сколько раз его сравнили на равенство
    comparisons = 0

    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        Probe.comparisons += 1
        return isinstance(other, Probe) and self.value == other.value

    def __lt__(self, other):
        return self.value < other.value

    def __hash__(self):
        return hash(self.value)

    def __repr__(self):
        return f"Probe({self.value})"


def probes(values):
    # список обёрток для последовательности чисел
    return [Probe(v) for v in values]


def count_comparisons(func, *args):
    # вызывает func(*args) и возвращает пару (результат, сколько было сравнений на равенство)
    Probe.comparisons = 0
    result = func(*args)
    return result, Probe.comparisons


print("инструменты для проверок загружены")

инструменты для проверок загружены


## Уровень 1. Обязательный

### Задача 1. Есть ли повторы

Верните `True`, если в списке есть хотя бы два равных элемента, и `False`, если все разные.
Это функция из разминки — только теперь за один проход.

```
has_duplicates([3, 1, 4, 1, 5])  ->  True
has_duplicates([2, 7, 1, 8])     ->  False
```

*Подсказка:* то, что уже встречалось, храните не в списке, а в множестве.

In [16]:
def has_duplicates(items: list) -> bool:
    """Есть ли в списке повторяющиеся элементы."""
    # ваш код здесь
    if len(items)==len(set(items)):
      return False
    return True
    pass


assert has_duplicates([3, 1, 4, 1, 5]) is True
assert has_duplicates([2, 7, 1, 8]) is False
assert has_duplicates([]) is False
assert has_duplicates([5]) is False
assert has_duplicates([5, 5]) is True
assert has_duplicates(["а", "б", "а"]) is True
assert has_duplicates([1, 1.0]) is True          # 1 == 1.0, это повтор

source = [1, 2, 1]
has_duplicates(source)
assert source == [1, 2, 1], "исходный список менять нельзя"

# способ: на тысяче разных значений не должно быть сотен тысяч сравнений
result, comparisons = count_comparisons(has_duplicates, probes(range(1000)))
assert result is False
assert comparisons <= 2000, f"сравнений {comparisons} — похоже на поиск перебором по списку"
result, comparisons = count_comparisons(has_duplicates, probes(list(range(1000)) + [999]))
assert result is True
assert comparisons <= 2000, f"сравнений {comparisons} — похоже на поиск перебором по списку"
print("Задача 1: ок")

Задача 1: ок


### Задача 2. Первые вхождения

Верните новый список, в котором каждое значение встречается один раз — на месте своего
первого появления.

```
unique([3, 1, 3, 2, 1])  ->  [3, 1, 2]
unique(["б", "а", "б"])  ->  ['б', 'а']
```

*Подсказка:* `list(set(items))` не годится — множество не помнит порядок. Нужны оба:
множество, чтобы быстро проверять «уже было?», и список для ответа.

In [17]:
def unique(items: list) -> list:
    """Значения без повторов в порядке первого появления."""
    # ваш код здесь
    a=set()
    b=[]
    for i in items:
      if i not in a:
        b.append(i)
        a.add(i)
    return b
    pass


assert unique([3, 1, 3, 2, 1]) == [3, 1, 2]
assert unique(["б", "а", "б"]) == ["б", "а"]
assert unique([]) == []
assert unique([7, 7, 7]) == [7]
assert unique([5, 4, 3, 2, 1]) == [5, 4, 3, 2, 1], "порядок первого появления должен сохраниться"
assert unique([1, True, 1.0, 2]) == [1, 2]       # True и 1.0 равны единице

source = [2, 2, 1]
unique(source)
assert source == [2, 2, 1], "исходный список менять нельзя"

result, comparisons = count_comparisons(unique, probes([i % 500 for i in range(1000)]))
assert [p.value for p in result] == list(range(500))
assert comparisons <= 2000, f"сравнений {comparisons} — похоже, «уже было?» проверяется по списку"
print("Задача 2: ок")

Задача 2: ок


### Задача 3. Общие буквы

Даны два слова. Верните пару строк: буквы, которые есть в обоих словах, и буквы, которые есть
ровно в одном из них. Каждая буква — один раз, буквы в строке — по алфавиту.

```
letters_report("кошка", "мышка")  ->  ('акш', 'моы')
letters_report("аб", "вг")        ->  ('', 'абвг')
```

*Подсказка:* две операции над множествами и `"".join(sorted(...))`.

In [18]:
def letters_report(a: str, b: str) -> tuple:
    """(буквы из обоих слов, буквы ровно из одного слова) — по алфавиту, без повторов."""
    # ваш код здесь
    aa=set(a)
    bb=set(b)
    c=sorted(list(aa^bb))
    d=sorted(list(aa&bb))
    cc="".join(c)
    dd="".join(d)


    return(dd,cc)
    pass


assert letters_report("кошка", "мышка") == ("акш", "моы")
assert letters_report("аб", "вг") == ("", "абвг")
assert letters_report("", "") == ("", "")
assert letters_report("аааа", "а") == ("а", ""), "каждая буква — один раз"
assert letters_report("кот", "ток") == ("кот", "")
assert letters_report("ба", "") == ("", "аб"), "буквы — по алфавиту"

rnd = random.Random(3)
for _ in range(300):
    a = "".join(rnd.choice("абвгд") for _ in range(rnd.randint(0, 6)))
    b = "".join(rnd.choice("абвгд") for _ in range(rnd.randint(0, 6)))
    both = "".join(ch for ch in "абвгд" if ch in a and ch in b)
    only_one = "".join(ch for ch in "абвгд" if (ch in a) != (ch in b))
    assert letters_report(a, b) == (both, only_one), (a, b)
print("Задача 3: ок")

Задача 3: ок


### Задача 4. Первый неповторяющийся

Верните первый символ строки, который встречается в ней ровно один раз. Если такого нет —
пустую строку.

```
first_unique("абракадабра")  ->  'к'
first_unique("ааббв")        ->  'в'
first_unique("абаб")         ->  ''
```

*Подсказка:* два прохода. Сначала посчитайте в словаре, сколько раз встречается каждый символ.
Потом — линейный поиск из видеолекции: первый символ, у которого счётчик равен единице.

In [19]:
def first_unique(text: str) -> str:
    """Первый символ, который встречается в строке ровно один раз, иначе ''."""
    # ваш код здесь
    a = {}

    for i in text:
        a[i] = a.get(i, 0) + 1
    for i in text:
        if a[i] == 1:

            return i
    return ""


assert first_unique("абракадабра") == "к"
assert first_unique("ааббв") == "в"
assert first_unique("абаб") == ""
assert first_unique("") == ""
assert first_unique("я") == "я"
assert first_unique("Аа") == "А", "большая и маленькая буквы — разные символы"
assert first_unique("а а") == " ", "пробел — тоже символ"

rnd = random.Random(4)
for _ in range(300):
    s = "".join(rnd.choice("абв") for _ in range(rnd.randint(0, 7)))
    expected = ""
    for ch in s:
        if s.count(ch) == 1:
            expected = ch
            break
    assert first_unique(s) == expected, s
print("Задача 4: ок")

Задача 4: ок


### Задача 5. Соседи на поле

Клетки поля заданы парами `(строка, столбец)`. Множество `occupied` — занятые клетки.
Верните, сколько из восьми соседей клетки `cell` заняты. Сама клетка не считается.

```
occupied = {(0, 0), (0, 1), (1, 1), (5, 5)}
count_neighbours(occupied, (1, 0))  ->  3
count_neighbours(occupied, (0, 0))  ->  2
```

*Подсказка:* перебирайте не занятые клетки, а восемь соседей: смещение по строке и по столбцу —
от −1 до 1, кроме `(0, 0)`. Проверка `(r, c) in occupied` стоит O(1).

In [20]:
def count_neighbours(occupied: set, cell: tuple) -> int:
    """Сколько из восьми соседних клеток занято."""
    # ваш код здесь
    r, c = cell
    k = 0
    for dr in (-1, 0, 1):

        for dc in (-1, 0, 1):


            if (dr, dc) != (0, 0) and (r + dr, c + dc) in occupied:
                k += 1
    return k

occupied = {(0, 0), (0, 1), (1, 1), (5, 5)}
assert count_neighbours(occupied, (1, 0)) == 3
assert count_neighbours(occupied, (0, 0)) == 2, "сама клетка не считается"
assert count_neighbours(occupied, (5, 5)) == 0
assert count_neighbours(set(), (0, 0)) == 0
assert count_neighbours({(-1, -1)}, (0, 0)) == 1
around = {(r, c) for r in (-1, 0, 1) for c in (-1, 0, 1)}
assert count_neighbours(around, (0, 0)) == 8

before = set(occupied)
count_neighbours(occupied, (0, 0))
assert occupied == before, "множество менять нельзя"


class NoIteration(set):
    # множество, которое не даёт перебирать себя целиком
    def __iter__(self):
        raise AssertionError("не перебирайте все занятые клетки — проверяйте восемь соседей")


big = NoIteration((r, c) for r in range(300) for c in range(300) if (r + c) % 3 == 0)
expected = 0
for dr in (-1, 0, 1):
    for dc in (-1, 0, 1):
        if (dr, dc) != (0, 0) and (150 + dr + 150 + dc) % 3 == 0:
            expected += 1
assert count_neighbours(big, (150, 150)) == expected
print("Задача 5: ок")

Задача 5: ок


### Задача 6. Набор домино

Костяшка домино — пара чисел. Костяшки `(2, 5)` и `(5, 2)` — одна и та же, её просто повернули.
Верните, сколько **разных** костяшек в наборе.

```
count_tiles([(2, 5), (5, 2), (3, 3)])  ->  2
count_tiles([(0, 1), (1, 2)])          ->  2
```

*Подсказка:* прежде чем класть пару в множество, приведите её к одному виду —
как с рукопожатиями на лекции.

In [21]:
def count_tiles(tiles: list) -> int:
    """Число разных костяшек: (a, b) и (b, a) — одна костяшка."""
    # ваш код здесь

    a=set()
    for i in tiles:
        a.add((max(i),min(i)))

    return(len(a))
    pass


assert count_tiles([(2, 5), (5, 2), (3, 3)]) == 2
assert count_tiles([(0, 1), (1, 2)]) == 2
assert count_tiles([]) == 0
assert count_tiles([(4, 4), (4, 4)]) == 1
assert count_tiles([(1, 2), (2, 1), (1, 2)]) == 1
assert count_tiles([(6, 0), (0, 6), (6, 6), (0, 0)]) == 3

rnd = random.Random(6)
for _ in range(300):
    tiles = [(rnd.randint(0, 3), rnd.randint(0, 3)) for _ in range(rnd.randint(0, 8))]
    expected = 0
    for i in range(len(tiles)):
        a, b = tiles[i]
        if (a, b) not in tiles[:i] and (b, a) not in tiles[:i]:
            expected += 1
    assert count_tiles(tiles) == expected, tiles
print("Задача 6: ок")

Задача 6: ок


## Уровень 2. На сочетание идей

### Задача 7. Взаимные подписки

Список `follows` — подписки в соцсети: пара `(a, b)` значит «a подписан на b». Верните
отсортированный список пар `(a, b)`, где `a < b` и оба подписаны друг на друга. Подписки
в списке могут повторяться; подписку на самого себя не учитываем.

```
mutual([("аня", "боря"), ("боря", "аня"), ("аня", "вера")])  ->  [('аня', 'боря')]
```

*Подсказка:* положите все подписки в множество — тогда «подписан ли b на a» проверяется
за O(1). Чтобы каждая взаимная пара попала в ответ один раз, кладите её в упорядоченном виде
во второе множество.

In [22]:
def mutual(follows: list) -> list:
    """Отсортированный список взаимных пар (a, b), где a < b."""
    # ваш код здесь
    a = set(follows)
    r = set()
    for x, y in a:
        if x < y and (y, x) in a:

            r.add((x, y))


    return sorted(r)

    pass







assert mutual([("аня", "боря"), ("боря", "аня"), ("аня", "вера")]) == [("аня", "боря")]
assert mutual([]) == []
assert mutual([("аня", "аня"), ("аня", "аня")]) == [], "подписка на себя не считается"
assert mutual([("аня", "боря"), ("аня", "боря"), ("боря", "аня")]) == [("аня", "боря")]
assert mutual([("в", "а"), ("а", "в"), ("б", "в"), ("в", "б")]) == [("а", "в"), ("б", "в")]

source = [("аня", "боря"), ("боря", "аня")]
mutual(source)
assert source == [("аня", "боря"), ("боря", "аня")], "исходный список менять нельзя"

rnd = random.Random(7)
people = ["аня", "боря", "вера", "гоша", "даня", "ева"]
for _ in range(200):
    follows = [(rnd.choice(people), rnd.choice(people)) for _ in range(rnd.randint(0, 15))]
    expected = []
    for a, b in follows:
        for c, d in follows:
            if a < b and c == b and d == a and (a, b) not in expected:
                expected.append((a, b))
    assert mutual(follows) == sorted(expected), follows
print("Задача 7: ок")

Задача 7: ок


### Задача 8. Анаграммы

Анаграммы — слова из одних и тех же букв в разном порядке: «кот», «ток», «окт». Разбейте список
слов на группы анаграмм. Группы идут в порядке первого появления, слова внутри группы —
в исходном порядке.

```
group_anagrams(["кот", "кит", "ток", "тик", "окт"])  ->  [['кот', 'ток', 'окт'], ['кит', 'тик']]
```

*Подсказка:* у всех анаграмм одинаковый «отпечаток» — отсортированные буквы. Это и есть
ключ словаря. Только `sorted` возвращает список, а список ключом быть не может: превратите
его в строку или кортеж. Порядок групп получится сам — словарь помнит порядок добавления.

In [23]:
def group_anagrams(words: list) -> list:
    """Список групп анаграмм в порядке первого появления."""
    # ваш код здесь

    groups = {}


    for i in words:
        key = "".join(sorted(i))
        if key not in groups:

            groups[key] = []
        groups[key].append(i)




    return list(groups.values())

    pass


assert group_anagrams(["кот", "кит", "ток", "тик", "окт"]) == [["кот", "ток", "окт"], ["кит", "тик"]]
assert group_anagrams([]) == []
assert group_anagrams(["а"]) == [["а"]]
assert group_anagrams(["абб", "ааб"]) == [["абб"], ["ааб"]], "одинаковый набор букв — ещё не анаграмма"
assert group_anagrams(["аб", "ба", "аб"]) == [["аб", "ба", "аб"]]
assert group_anagrams(["абв", "аб"]) == [["абв"], ["аб"]]

source = ["ток", "кот"]
group_anagrams(source)
assert source == ["ток", "кот"], "исходный список менять нельзя"

rnd = random.Random(8)
for _ in range(200):
    words = ["".join(rnd.choice("абв") for _ in range(rnd.randint(1, 3)))
             for _ in range(rnd.randint(0, 10))]
    expected = []
    for w in words:
        for group in expected:
            if sorted(group[0]) == sorted(w):
                group.append(w)
                break
        else:
            expected.append([w])
    assert group_anagrams(words) == expected, words
print("Задача 8: ок")

Задача 8: ок


### Задача 9. Кто не сдал

Учитель собирает работы. `students` — список класса (имена не повторяются), `submitted` — имена
с подписанных работ. В `submitted` бывают повторы (сдал дважды) и опечатки (такого имени
в классе нет). Верните пару: кто не сдал — в порядке списка класса, и неизвестные имена —
без повторов, по алфавиту.

```
report(["Аня", "Боря", "Вера"], ["Вера", "Бора", "Вера"])  ->  (['Аня', 'Боря'], ['Бора'])
```

*Подсказка:* два множества и одно включение. `x in список` внутри цикла — это снова O(n²).

In [24]:
def report(students: list, submitted: list) -> tuple:
    """(не сдавшие в порядке списка класса, неизвестные имена по алфавиту)."""
    # ваш код здесь

    a = set(students)

    b = set(submitted)
    m=[]
    for s in students:
      if s not in b:
        m.append(s)

    c = sorted(b-a)

    return (m, c)

    pass


assert report(["Аня", "Боря", "Вера"], ["Вера", "Бора", "Вера"]) == (["Аня", "Боря"], ["Бора"])
assert report([], []) == ([], [])
assert report(["Аня"], ["Аня", "Аня"]) == ([], [])
assert report(["Яна", "Аня"], []) == (["Яна", "Аня"], []), "порядок — как в списке класса"
assert report([], ["Юра", "Ася", "Юра"]) == ([], ["Ася", "Юра"])

students, submitted = ["Аня", "Боря"], ["Вася", "Аня"]
report(students, submitted)
assert students == ["Аня", "Боря"] and submitted == ["Вася", "Аня"], "списки менять нельзя"

(missing, unknown), comparisons = count_comparisons(report, probes(range(1000)), probes(range(500, 1500)))
assert [p.value for p in missing] == list(range(500))
assert [p.value for p in unknown] == list(range(1000, 1500))
assert comparisons <= 4000, f"сравнений {comparisons} — похоже на поиск перебором по списку"
print("Задача 9: ок")

Задача 9: ок


### Задача 10. Указатель слов

Постройте указатель: для каждого слова — список номеров строк (нумерация с единицы),
где оно встречается. Слова разделены пробелами, регистр не учитываем. Номер строки попадает
в список один раз, даже если слово встретилось в строке несколько раз.

```
build_index(["кот спит", "кот ест кот", "пёс спит"])
    ->  {'кот': [1, 2], 'спит': [1, 3], 'ест': [2], 'пёс': [3]}
```

*Подсказка:* словарь списков, как при группировке. Чтобы номер строки не попал дважды,
пройдите не по словам строки, а по множеству её слов.

In [25]:
def build_index(lines: list) -> dict:
    """Слово (в нижнем регистре) -> номера строк по возрастанию, без повторов."""
    # ваш код здесь

    res = {}


    for i in range(len(lines)):

        for w in set(lines[i].lower().split()):

            if w not in res:

                res[w] = []
            res[w].append(i+1)
    return res
    pass


assert build_index(["кот спит", "кот ест кот", "пёс спит"]) == {
    "кот": [1, 2], "спит": [1, 3], "ест": [2], "пёс": [3]
}
assert build_index([]) == {}
assert build_index(["", "   "]) == {}
assert build_index(["Кот КОТ кот"]) == {"кот": [1]}, "регистр не учитываем, номер строки — один раз"
assert build_index(["а", "", "а"]) == {"а": [1, 3]}, "нумерация с единицы, пустые строки тоже считаются"

rnd = random.Random(10)
for _ in range(200):
    lines = [" ".join(rnd.choice(["а", "б", "в", "А"]) for _ in range(rnd.randint(0, 4)))
             for _ in range(rnd.randint(0, 6))]
    words = set()
    for line in lines:
        for w in line.lower().split():
            words.add(w)
    expected = {}
    for w in words:
        expected[w] = [i + 1 for i in range(len(lines)) if w in lines[i].lower().split()]
    assert build_index(lines) == expected, lines
print("Задача 10: ок")

Задача 10: ок


## Уровень 3. Со звёздочкой

### Задача 11. Своя хеш-таблица

Соберите свою хеш-таблицу — множество на корзинах, как на лекции. Таблица — это словарь с двумя
полями: `size` — сколько в ней элементов, `buckets` — список корзин. Функция `new_table`
уже написана.

Напишите `table_add(table, x)` — добавить `x`, если его ещё нет, — и `table_contains(table, x)`.
Когда после добавления элементов становится больше двух третей от числа корзин, таблицу нужно
расширить вдвое: завести вдвое больше пустых корзин и разложить по ним все элементы заново.

```
table = new_table()
for word in ["кот", "кит", "кот"]:
    table_add(table, word)
table["size"]                  ->  2
table_contains(table, "кит")   ->  True
table_contains(table, "пёс")   ->  False
```

*Подсказка:* номер корзины — `hash(x) % len(table["buckets"])`. После расширения номера
у всех элементов меняются, поэтому раскладывать надо заново, а не копировать корзины.

In [26]:
def new_table() -> dict:
    """Пустая таблица на 8 корзин."""
    return {"size": 0, "buckets": [[] for _ in range(8)]}


def table_add(table: dict, x) -> None:
    """Добавляет x в таблицу, если его там ещё нет."""
    # ваш код здесь
    pass


def table_contains(table: dict, x) -> bool:
    """Есть ли x в таблице."""
    # ваш код здесь
    pass


table = new_table()
for word in ["кот", "кит", "кот"]:
    table_add(table, word)
assert table["size"] == 2
assert table_contains(table, "кит") is True
assert table_contains(table, "пёс") is False
assert table_contains(new_table(), 0) is False

# сверяем с настоящим множеством на случайных числах
rnd = random.Random(11)
table, real = new_table(), set()
for _ in range(3000):
    x = rnd.randint(-2000, 2000)
    table_add(table, x)
    real.add(x)
    assert table["size"] == len(real), "size должен совпадать с числом разных элементов"
    assert table["size"] * 3 <= len(table["buckets"]) * 2, \
        "таблица заполнена больше чем на две трети — пора было расширять"
stored = sorted(x for bucket in table["buckets"] for x in bucket)
assert stored == sorted(real), "в корзинах должны лежать ровно добавленные элементы, без повторов"
for x in range(-2100, 2100):
    assert table_contains(table, x) == (x in real), x

# каждый элемент лежит в той корзине, где его будут искать
n = len(table["buckets"])
for i, bucket in enumerate(table["buckets"]):
    for x in bucket:
        assert hash(x) % n == i, f"{x} лежит не в своей корзине — после расширения раскладывайте заново"

# корзин становится вдвое больше: 8, 16, 32, ...
size = 8
while size < n:
    size *= 2
assert size == n, f"корзин {n} — их число должно получаться удвоением из 8"

# проверка смотрит в одну корзину, а не во всю таблицу
table = new_table()
for p in probes(range(1000)):
    table_add(table, p)
Probe.comparisons = 0
for p in probes(range(1000)):
    assert table_contains(table, p)
assert Probe.comparisons <= 3000, f"сравнений {Probe.comparisons} — похоже, поиск идёт не в одной корзине"
print("Задача 11: ок")

AssertionError: 

### Задача 12. Кеш на последние запросы

Кеш помнит последние `capacity` разных запросов. Когда приходит запрос:

- если он есть в кеше — это **попадание**, и запрос становится самым свежим;
- если нет — **промах**: запрос добавляется в кеш как самый свежий, а если кеш переполнился,
  из него выбрасывается самый давний.

Верните число промахов.

```
cache_misses(2, ["a", "b", "a", "c", "b"])  ->  4
```

Разбор примера: `a` — промах, `b` — промах, `a` — попадание (теперь `a` свежее `b`),
`c` — промах, выбрасываем `b`; `b` — промах.

*Подсказка:* словарь помнит порядок добавления. Чтобы сделать ключ самым свежим, удалите его
и добавьте снова. Самый давний — первый ключ: `next(iter(cache))`. Все операции — O(1).

In [ ]:
def cache_misses(capacity: int, requests: list) -> int:
    """Число промахов кеша, который помнит capacity последних разных запросов."""
    # ваш код здесь

    c={}
    m=0
    for r in requests:
        if r in c:
            del c[r]
        else:

            m+=1


        c[r] = True
        if len(c)> capacity:

            del c[list(c)[0]]


    return m
    pass


assert cache_misses(2, ["a", "b", "a", "c", "b"]) == 4
assert cache_misses(2, []) == 0
assert cache_misses(0, ["a", "a", "a"]) == 3, "кеш на ноль запросов ничего не помнит"
assert cache_misses(1, ["a", "a", "a"]) == 1
assert cache_misses(1, ["a", "b", "a", "b"]) == 4
assert cache_misses(3, ["a", "b", "c", "a", "d", "b"]) == 5, "освежённый запрос не выбрасывается первым"


def slow_misses(capacity, requests):
    # медленно, зато очевидно правильно: кеш — список от давнего к свежему
    cache, misses = [], 0
    for r in requests:
        if r in cache:
            cache.remove(r)
        else:
            misses += 1
        cache.append(r)
        if len(cache) > capacity:
            cache.pop(0)
    return misses


rnd = random.Random(12)
for _ in range(300):
    capacity = rnd.randint(0, 4)
    requests = [rnd.choice("абвгде") for _ in range(rnd.randint(0, 20))]
    assert cache_misses(capacity, requests) == slow_misses(capacity, requests), (capacity, requests)
print("Задача 12: ок")

## Что дальше

Следующая пара — **функции**: параметры и значения по умолчанию, произвольное число аргументов
и ловушка с изменяемым значением по умолчанию.

Если хочется забежать вперёд: функция `def add(item, box=[])` дописывает `item` в `box`
и возвращает его. Вызовите её два раза подряд без второго аргумента и попробуйте объяснить
результат, опираясь на занятие про ссылки и изменяемость.